---
# <div style="text-align: center"> Introduction </div>
---

Along the previous tutorials, <span style="color:blue">**SCOPE**</span> interacts with the different parts of the code to handle the execution of computational workflows. 

These are the topics covered in each tutorial:
1) The **System** class and its sources: the **Specie**, **Cell** and **Atom** classes  
2) The Computational workflow: **Branch**, **Workflow**, **Job**, and **Computation** classes  
3) The **State** class  
4) The **Data**, **Collection** and **VNM** classes
5) The **Input_data** class, and **scope input files**
6) Running <span style="color:blue">**SCOPE**</span> - Part 1: File Structure
7) Running <span style="color:blue">**SCOPE**</span> - Part 2: Execution 
8) Running <span style="color:blue">**SCOPE**</span> - Part 3: Detailed Actions

---
# <div style="text-align: center"> Azo-Tutorial 1: Subclasses derived from SCOPE </div> 
---

## Part 1. Navigate an existing SYSTEM

In [1]:
import os
import scope 
from scope_azo.azo_classes import *
from scope_azo.azo_functions import * 

In [2]:
tutorial_folder = '../Data/Azo/'

In [3]:
azo_sys = load_binary(f'{tutorial_folder}/Systems/Azobenzene/Azobenzene.npy')

In [4]:
found, sou = azo_sys.find_source('TSrot_A_T')
print(sou)

---------- SCOPE Molecule_azo Object ------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule_azo
 Name                  = tsrot_a_t
 Number of Atoms       = 24
 Formula               = H10-C12-N2
 Charge                = 0
 Spin (alpha - beta)   = 2
 Number of Parents     = 0
 Has Adjacency Matrix  = NO 
 Has Bonds             = NO
-------------------------------------------------



In [5]:
azo_sys.branches

[]

In [13]:
azo_sys.remove_branch("tutorial_azo")

In [7]:
azo_sys 

------------- SCOPE System_azo Object -------
 Name                  = Azobenzene
 Atom Indices for Dihedral = [1, 0, 6, 7, 8, 9]
 Version               = 1.0
 Type                  = system
 Subtype               = system_azo
 Source Path           = /home/mserrano/scope/Scope_Tutorials/Data/Azo/Sources/Azobenzene/
 System File Path      = /home/mserrano/scope/Scope_Tutorials/Data/Azo/Systems/Azobenzene/
 System File Name      = /home/mserrano/scope/Scope_Tutorials/Data/Azo/Systems/Azobenzene/Azobenzene.npy
 Computations Path     = /home/mserrano/scope/Scope_Tutorials/Data/Azo/Computations/Azobenzene/

 Num Sources           = 8
     idx: type, name, formula               
     0: specie, trans, H10-C12-N2 
     1: specie, tsrot_a_s, H10-C12-N2 
     2: specie, tsrot_a_t, H10-C12-N2 
     3: specie, tsrot_b_s, H10-C12-N2 
     4: specie, tsrot_b_t, H10-C12-N2 
     5: specie, tsinv_l, H10-C12-N2 
     6: specie, tsinv_r, H10-C12-N2 
     7: specie, cis, H10-C12-N2 
-------------------

In [8]:
azo_sys.save()

In [9]:
found, branch = azo_sys.find_branch("tutorial_azo")
print(found)
found, workflow = branch.find_workflow("TSrot_A_T")
print(found)

found, job = workflow.find_job('pbe_opt')
print(job)

False


AttributeError: 'NoneType' object has no attribute 'find_workflow'

In [8]:
azo_sys.find_computation(branch_keyword="tutorial_azo", workflow_keyword='TSrot_A_T', job_keyword='pbe opt', comp_keyword='')

AttributeError: 'Branch' object has no attribute 'keyword'

In [4]:
azo_sys.sources[-2].states[-1].results

{'energy': energy: -565.01177962 au}

In [10]:
## See Tutorial 5 for more information

task1_input = """
&environment
   max_jobs         = 24
   max_procs        = 96
   requested_procs  = 12
/

&options
/

&job_data
   branch       = 'tutorial_azo'
   workflow     = 'all'
   job_setup    = 'regular'
   suffix       = 'scf'
   keyword      = 'pbe scf'

   istate       = 'initial'
   fstate       = 'pbe_initial'

   hierarchy    = 1
   requisites   = []
   constrains   = ['self']
   must_be_good = True
/

&qc_data
   software     = 'g16'
   jobtype      = 'opt'
   functional   = 'pbe'
   basis        = 'sto-3g'
   is_grimme    = True
   loose_opt    = True
/"""


## See Tutorial 5 for more information

task2_input = """
&environment
   max_jobs         = 24
   max_procs        = 96
   requested_procs  = 12
/

&options
/

&job_data
   branch       = 'tutorial_azo'
   workflow     = 'all'
   job_setup    = 'regular'
   suffix       = 'opt'
   keyword      = 'pbe opt'

   istate       = 'pbe_initial'
   fstate       = 'pbe_opt'

   hierarchy    = 2
   requisites   = ['pbe scf']
   constrains   = ['self']
   must_be_good = True
/

&qc_data
   software     = 'g16'
   jobtype      = 'opt'
   functional   = 'pbe'
   basis        = 'sto-3g'
   is_grimme    = True
   loose_opt    = True
/"""

In [11]:
file_name1 = os.path.join(tutorial_folder, "task1.scope")
file_name2 = os.path.join(tutorial_folder, "task2.scope")

scope.read_write.save_text(task1_input, file_name1)
scope.read_write.save_text(task2_input, file_name2)